# Phase 1 — Zero-shot Baseline (v0)

**Mục đích:** Đo khả năng ước giá của `Qwen/Qwen3.5-4B-Base` chưa fine-tune trên 200 test samples.  
Kết quả là lower bound (v0) để đo improvement sau các phase training.

**Config chốt từ Phase 0:**
- Model: `Qwen/Qwen3.5-4B-Base` — Base, 4-bit NF4 (PEFT + bitsandbytes)
- Dataset: `SeanSunny/items_prompts_tv_3` — test split, 200 random samples (seed=42)
- `max_seq_length = 192` | `max_new_tokens = 4`
- `pred_vnd = predict(prompt) * 1000` (completion là đơn vị nghìn đồng)
- Output: `results/v0_results.json`

**Môi trường:** CUDA 12.8 | PyTorch 2.9.0+cu128 | RTX 3090 Ti (25.3 GB VRAM) | Compute 8.6

**Ghi chú Unsloth:** `Qwen3.5-4B-Base` có Vision Encoder trong architecture (Hybrid: Gated DeltaNet + sparse MoE).  
Unsloth 2026.4.8 load đúng kiến trúc VL nhưng không hỗ trợ text-only inference ổn định.  
Phase 1 dùng HF transformers + bitsandbytes. Phase 2+ sẽ thử `unsloth/Qwen3.5-4B-Base` (Unsloth repo).

**Kỳ vọng:** RMSLE > 1.0 — model chưa biết pattern completion số thuần. Đây là expected.

## 0. Cài đặt (chạy một lần sau khi git clone + uv sync)

```bash
uv sync
uv add unsloth          # optional, dung cho Phase 2+
uv add "transformers>=5.2.0"
```

Phase 1 chi can: `transformers`, `bitsandbytes`, `accelerate`, `datasets`, `python-dotenv` — tat ca da co trong pyproject.toml.

In [ ]:
# Chi chay neu chua uv sync
# !uv add "transformers>=5.2.0" bitsandbytes accelerate datasets python-dotenv

In [ ]:
import os
import re
import sys
import json
import time
import numpy as np
from tqdm import tqdm
from pathlib import Path

import torch
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login
from sklearn.metrics import mean_absolute_error, r2_score
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

NOTEBOOK_DIR = Path("__file__").parent if "__file__" in dir() else Path(".")
sys.path.insert(0, str(NOTEBOOK_DIR))

print("Imports OK")
print(f"torch : {torch.__version__}")
import transformers
print(f"transformers : {transformers.__version__}")

In [ ]:
BASE_MODEL    = "Qwen/Qwen3.5-4B-Base"
DATASET_NAME  = "SeanSunny/items_prompts_tv_3"

MAX_NEW_TOKENS = 4
EVAL_SAMPLES   = 200
SEED           = 42

RESULTS_DIR  = NOTEBOOK_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
RESULTS_FILE = RESULTS_DIR / "v0_results.json"

print(f"BASE_MODEL    : {BASE_MODEL}")
print(f"DATASET_NAME  : {DATASET_NAME}")
print(f"MAX_NEW_TOKENS: {MAX_NEW_TOKENS}")
print(f"EVAL_SAMPLES  : {EVAL_SAMPLES}")
print(f"RESULTS_FILE  : {RESULTS_FILE}")

In [ ]:
assert torch.cuda.is_available(), "GPU khong kha dung."
print(f"GPU   : {torch.cuda.get_device_name(0)}")
print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
cap = torch.cuda.get_device_capability()
print(f"Cap   : {cap} -> bf16={'yes' if cap[0] >= 8 else 'no'}")

In [ ]:
env_path = NOTEBOOK_DIR.parent / ".env"
load_dotenv(env_path)

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(HF_TOKEN)
    print(f"HF login OK (from {env_path})")
else:
    print(f"HF_TOKEN not found in {env_path}")
    login()

## 1. Load model — 4-bit NF4 (bitsandbytes)

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## 2. Load dataset — 200 random test samples

In [ ]:
dataset = load_dataset(DATASET_NAME)
test_sample = dataset["test"].shuffle(seed=SEED).select(range(EVAL_SAMPLES))

print(f"Test full  : {len(dataset['test']):,} items")
print(f"Eval sample: {len(test_sample):,} items")
print()
print("Sample item:")
print(f"  prompt[:80] : {test_sample[0]['prompt'][:80]!r}")
print(f"  completion  : {test_sample[0]['completion']!r}")
print(f"  price_vnd   : {test_sample[0]['price_vnd_true']:,}")

## 3. Predict function

Phiên bản đơn giản cho zero-shot — chưa dùng StoppingCriteria.  
Completion là đơn vị nghìn VND (ví dụ "150" = 150,000 VND), nên `pred_vnd = pred_k * 1000`.

In [ ]:
def predict_one(prompt: str) -> tuple[int, str]:
    """Returns (pred_thousands_vnd, raw_generated_text)."""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    prompt_len = inputs["input_ids"].shape[1]
    generated = tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True)
    match = re.search(r"\d+", generated)
    pred_k = int(match.group()) if match else 0
    return pred_k, generated


# Test 1 sample
t0 = time.time()
pred_k, raw = predict_one(test_sample[0]["prompt"])
elapsed = time.time() - t0

print(f"Raw output  : {raw!r}")
print(f"pred_k      : {pred_k}")
print(f"pred_vnd    : {pred_k * 1000:,}")
print(f"true_vnd    : {test_sample[0]['price_vnd_true']:,}")
print(f"Time/item   : {elapsed:.2f}s")

## 4. Inference — 200 samples

In [ ]:
preds_vnd   = []
trues_vnd   = []
raw_outputs = []

t_start = time.time()

for item in tqdm(test_sample, desc="Zero-shot inference"):
    pred_k, raw = predict_one(item["prompt"])
    preds_vnd.append(pred_k * 1000)
    trues_vnd.append(item["price_vnd_true"])
    raw_outputs.append(raw)

t_total = time.time() - t_start
print(f"\nDone: {EVAL_SAMPLES} samples in {t_total:.1f}s ({t_total / EVAL_SAMPLES:.2f}s/item)")
print(f"Items where model returned 0 (no digit): {preds_vnd.count(0)}")

## 5. Metrics

In [ ]:
from utils.evaluator import compute_metrics

y_true = np.array(trues_vnd, dtype=float)
y_pred = np.array(preds_vnd, dtype=float)

metrics = compute_metrics(y_true, y_pred)

print("=" * 45)
print(f"v0 Zero-shot — {EVAL_SAMPLES} test samples")
print("=" * 45)
print(f"RMSLE : {metrics['rmsle']:.4f}  (primary)")
print(f"MAE   : {metrics['mae']:,.0f} VND")
print(f"MAPE  : {metrics['mape']:.1f}%")
print(f"R2    : {metrics['r2']:.4f}")
print("=" * 45)
print("Day 4 v8 reference: RMSLE=0.4004")
print(f"Gap v0 vs v8: {metrics['rmsle'] - 0.4004:+.4f}")

## 6. Save results

In [ ]:
samples_out = []
for i in range(20):
    true_vnd = trues_vnd[i]
    pred_vnd = preds_vnd[i]
    err_pct = abs(pred_vnd - true_vnd) / true_vnd * 100 if true_vnd > 0 else None
    samples_out.append({
        "idx"           : i,
        "prompt_excerpt" : test_sample[i]["prompt"][:120],
        "generated_raw"  : raw_outputs[i],
        "pred_vnd"       : pred_vnd,
        "true_vnd"       : true_vnd,
        "error_pct"      : round(err_pct, 1) if err_pct is not None else None,
    })

results = {
    "version"        : "v0_zero_shot",
    "model"          : BASE_MODEL,
    "dataset"        : DATASET_NAME,
    "eval_samples"   : EVAL_SAMPLES,
    "seed"           : SEED,
    "max_new_tokens" : MAX_NEW_TOKENS,
    "inference_sec"  : round(t_total, 1),
    "sec_per_item"   : round(t_total / EVAL_SAMPLES, 2),
    "zero_pred_count": preds_vnd.count(0),
    "metrics"        : metrics,
    "samples"        : samples_out,
}

with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Saved: {RESULTS_FILE}")

In [ ]:
print(f"{'#':>3}  {'true_vnd':>12}  {'pred_vnd':>12}  {'err%':>8}  raw")
print("-" * 70)
for s in samples_out:
    err_str = f"{s['error_pct']:>7.1f}%" if s["error_pct"] is not None else "    N/A"
    print(f"{s['idx']:>3}  {s['true_vnd']:>12,}  {s['pred_vnd']:>12,}  {err_str}  {repr(s['generated_raw'])[:25]}")

## Leaderboard Day 5

In [ ]:
print(f"{'Version':<22} {'RMSLE':>8} {'MAE':>14} {'MAPE':>8} {'R2':>8}")
print("-" * 65)
print(f"{'v0 zero-shot':<22} {metrics['rmsle']:>8.4f} {metrics['mae']:>14,.0f} {metrics['mape']:>7.1f}% {metrics['r2']:>8.4f}")
print(f"{'v8 Day4 (ref)':<22} {'0.4004':>8} {'79,853':>14} {'30.7%':>8} {'0.692':>8}")
print()
print("Buoc tiep: Phase 2 — Smoke test v1 (03_train_v1_smoke.ipynb)")
print("  Confirm voi user truoc khi chay.")